In [13]:
import os
import numpy as np
import pandas as pd
from neuroHarmonize import harmonizationLearn
from neuroHarmonize import harmonizationApply

In [14]:
root = os.getcwd().replace("/code","")
tsv = pd.read_csv(f"{root}/all_participants_with_indices.tsv", sep="\t").iloc[:,1:]
tsv

,participant_id,age,sex,group,site,psqi_score,kss_score_baseline,kss_score_tsd,kss_score_psd,kss_score_recovery,...,dti-alps-baseline,dti-alps-sleep-deprivation,kss_score_sd,handedness,kss_scre_sd,pvt_meanrt_sd,n_back_baseline,n_back_sd,ksqi_score,first_session
0,sub-105006,26.063014,M,4.0,Juelich,NaN,3.0,8.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,sub-105009,22.808219,F,4.0,Juelich,NaN,3.0,5.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,sub-105014,26.265753,M,7.0,Juelich,NaN,NaN,9.0,6.0,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,sub-105030,26.780822,F,7.0,Juelich,NaN,NaN,9.0,6.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,sub-105036,21.498630,M,7.0,Juelich,NaN,NaN,8.0,4.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
313,sub-320,22.000000,M,3.0,Melbourne,3.0,2.5,NaN,NaN,NaN,...,NaN,NaN,3.5,NaN,NaN,2.099772,NaN,NaN,NaN,SD
314,sub-321,29.000000,F,3.0,Melbourne,6.0,3.0,NaN,NaN,NaN,...,NaN,NaN,8.0,NaN,NaN,2.612785,NaN,NaN,NaN,SD
315,sub-322,23.000000,M,3.0,Melbourne,4.0,2.0,NaN,NaN,NaN,...,NaN,NaN,4.0,NaN,NaN,2.735884,NaN,NaN,NaN,NN
316,sub-323,21.000000,M,3.0,Melbourne,5.0,2.0,NaN,NaN,NaN,...,NaN,NaN,5.0,NaN,NaN,2.045471,NaN,NaN,NaN,NN


In [15]:
# Create unique sub-num 
for row in tsv.index:
    sub = tsv.iloc[row,:].participant_id
    site = tsv.iloc[row,:].site

    prefix = site[0:2].upper()
    suffix = sub.replace("sub-","")
    
    new_sub = f"sub-{prefix}{suffix}"
    tsv.iloc[row,0] = new_sub

tsv

,participant_id,age,sex,group,site,psqi_score,kss_score_baseline,kss_score_tsd,kss_score_psd,kss_score_recovery,...,dti-alps-baseline,dti-alps-sleep-deprivation,kss_score_sd,handedness,kss_scre_sd,pvt_meanrt_sd,n_back_baseline,n_back_sd,ksqi_score,first_session
0,sub-JU105006,26.063014,M,4.0,Juelich,NaN,3.0,8.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,sub-JU105009,22.808219,F,4.0,Juelich,NaN,3.0,5.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,sub-JU105014,26.265753,M,7.0,Juelich,NaN,NaN,9.0,6.0,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,sub-JU105030,26.780822,F,7.0,Juelich,NaN,NaN,9.0,6.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,sub-JU105036,21.498630,M,7.0,Juelich,NaN,NaN,8.0,4.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
313,sub-ME320,22.000000,M,3.0,Melbourne,3.0,2.5,NaN,NaN,NaN,...,NaN,NaN,3.5,NaN,NaN,2.099772,NaN,NaN,NaN,SD
314,sub-ME321,29.000000,F,3.0,Melbourne,6.0,3.0,NaN,NaN,NaN,...,NaN,NaN,8.0,NaN,NaN,2.612785,NaN,NaN,NaN,SD
315,sub-ME322,23.000000,M,3.0,Melbourne,4.0,2.0,NaN,NaN,NaN,...,NaN,NaN,4.0,NaN,NaN,2.735884,NaN,NaN,NaN,NN
316,sub-ME323,21.000000,M,3.0,Melbourne,5.0,2.0,NaN,NaN,NaN,...,NaN,NaN,5.0,NaN,NaN,2.045471,NaN,NaN,NaN,NN


In [17]:
df = tsv.loc[tsv.index.repeat(2)].reset_index(drop=True) # Duplicate rows
df['SD'] = [0,1] * (int(len(df)/2)) # Add SD condition

# KSS
df['KSS'] = df['kss_score_baseline'].fillna(df['kss_score_recovery']) # Init KSS column
df.loc[df.SD == 1, "KSS"] = np.nan # Set SD connection as NaN
df['kss_score_sd'] = df['kss_score_sd'].fillna(df['kss_score_tsd']).fillna(df['kss_scre_sd'])
df['KSS'].fillna(df['kss_score_sd'], inplace=True)
 
# Chronotype
df["chronotype"].fillna(3, inplace=True) # Set missing chronotypes to intermediate
df["CHRONO_MORNING"] = 0 # Init morningness column
df["CHRONO_EVENING"] = 0 # Init eveningness column
df.loc[df.chronotype == 1, "CHRONO_MORNING"] = 1
df.loc[df.chronotype == 2, "CHRONO_EVENING"] = 1

# PVSVF
df.dropna(subset=["pvsvf-baseline", "pvsvf-sleep-deprivation"], inplace=True) # Drop subjects with NA
df['PVSVF'] = df["pvsvf-baseline"] # Init PVSVF column
df.loc[df.SD == 1, "PVSVF"] = np.nan # Set SD connection as NaN
df['PVSVF'].fillna(df["pvsvf-sleep-deprivation"], inplace=True)

# CPVF
df.dropna(subset=["cpvf-baseline", "cpvf-sleep-deprivation"], inplace=True) # Drop subjects with NA
df['CPVF'] = df["cpvf-baseline"] # Init CPVF column
df.loc[df.SD == 1, "CPVF"] = np.nan # Set SD connection as NaN
df['CPVF'].fillna(df["cpvf-sleep-deprivation"], inplace=True)

# RFWI
col_a = "rfwi-baseline"
col_b = "rfwi-sleep-deprivation"
df.loc[(df[col_a].isna()) ^ (df[col_b].isna()), [col_a, col_b]] = np.nan
df['RFWI'] = df[col_a] # Init RWFI column
df.loc[df.SD == 1, "RFWI"] = np.nan # Set SD connection as NaN
df['RFWI'].fillna(df[col_b], inplace=True)

# DTI-ALPS
col_a = "dti-alps-baseline"
col_b = "dti-alps-sleep-deprivation"
df.loc[(df[col_a].isna()) ^ (df[col_b].isna()), [col_a, col_b]] = np.nan
df['ALPS'] = df[col_a] # Init ALPS column
df.loc[df.SD == 1, "ALPS"] = np.nan # Set SD connection as NaN
df['ALPS'].fillna(df[col_b], inplace=True)
df.to_csv(f"{root}/all_participants_clean.csv")
df

,participant_id,age,sex,group,site,psqi_score,kss_score_baseline,kss_score_tsd,kss_score_psd,kss_score_recovery,...,ksqi_score,first_session,SD,KSS,CHRONO_MORNING,CHRONO_EVENING,PVSVF,CPVF,RFWI,ALPS
0,sub-JU105006,26.063014,M,4.0,Juelich,NaN,3.0,8.0,NaN,NaN,...,NaN,NaN,0,3.0,0,0,0.000291,0.000348,NaN,NaN
1,sub-JU105006,26.063014,M,4.0,Juelich,NaN,3.0,8.0,NaN,NaN,...,NaN,NaN,1,8.0,0,0,0.000283,0.000295,NaN,NaN
2,sub-JU105009,22.808219,F,4.0,Juelich,NaN,3.0,5.0,NaN,NaN,...,NaN,NaN,0,3.0,0,0,0.000206,0.000289,NaN,NaN
3,sub-JU105009,22.808219,F,4.0,Juelich,NaN,3.0,5.0,NaN,NaN,...,NaN,NaN,1,5.0,0,0,0.000205,0.000317,NaN,NaN
10,sub-JU105041,21.019178,F,4.0,Juelich,NaN,2.0,3.0,NaN,NaN,...,NaN,NaN,0,2.0,0,0,0.000190,0.000387,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
631,sub-ME322,23.000000,M,3.0,Melbourne,4.0,2.0,NaN,NaN,NaN,...,NaN,NN,1,4.0,1,0,0.001028,0.000253,NaN,NaN
632,sub-ME323,21.000000,M,3.0,Melbourne,5.0,2.0,NaN,NaN,NaN,...,NaN,NN,0,2.0,0,0,0.000752,0.000251,NaN,NaN
633,sub-ME323,21.000000,M,3.0,Melbourne,5.0,2.0,NaN,NaN,NaN,...,NaN,NN,1,5.0,0,0,0.000774,0.000237,NaN,NaN
634,sub-ME324,21.000000,F,3.0,Melbourne,9.0,3.0,NaN,NaN,NaN,...,NaN,SD,0,3.0,0,0,0.000590,0.000275,NaN,NaN


In [11]:
harm = pd.DataFrame({"SUB" : df["participant_id"],
                     "SITE" : df["site"],
                     "SD" : df["SD"],
                     "AGE" : df["age"],
                     "SEX_MALE" : df["sex"],
                     "KSS" : df["KSS"],
                     "CHRONO_MORNING" : df["CHRONO_MORNING"],
                     "CHRONO_EVENING" : df["CHRONO_EVENING"],
                     "PVSVF" : df["PVSVF"],
                     "CPVF" : df["CPVF"],
                     "RFWI" : df["RFWI"],
                     "ALPS" : df["ALPS"]})
harm.set_index("SUB", inplace=True)
harm.SEX_MALE = harm.SEX_MALE.map({"M" : 1, "F" : 0})
harm.to_csv(f"{root}/all_participants_model.csv")
harm

,SITE,SD,AGE,SEX_MALE,KSS,CHRONO_MORNING,CHRONO_EVENING,PVSVF,CPVF,RFWI,ALPS
SUB,,,,,,,,,,,
sub-JU105006,Juelich,0,26.063014,1,3.0,0,0,0.000291,0.000348,NaN,NaN
sub-JU105006,Juelich,1,26.063014,1,8.0,0,0,0.000283,0.000295,NaN,NaN
sub-JU105009,Juelich,0,22.808219,0,3.0,0,0,0.000206,0.000289,NaN,NaN
sub-JU105009,Juelich,1,22.808219,0,5.0,0,0,0.000205,0.000317,NaN,NaN
sub-JU105041,Juelich,0,21.019178,0,2.0,0,0,0.000190,0.000387,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
sub-ME322,Melbourne,1,23.000000,1,4.0,1,0,0.001028,0.000253,NaN,NaN
sub-ME323,Melbourne,0,21.000000,1,2.0,0,0,0.000752,0.000251,NaN,NaN
sub-ME323,Melbourne,1,21.000000,1,5.0,0,0,0.000774,0.000237,NaN,NaN


In [181]:
anat_features = ["PVSVF"]
dti_features = ["ALPS"]

In [197]:
anat = harm.drop(columns=dti_features)
anat = anat.dropna(subset=anat_features)
anat_array = np.array(anat[anat_features])
anat_covars = anat.drop(columns=anat_features)
print(f"Array of {anat_array.shape} and covars of {anat_covars.shape}")

Array of (566, 1) and covars of (566, 6)


In [198]:
dti = harm.drop(columns=anat_features)
dti = dti.dropna(subset=dti_features)
dti_array = np.array(dti[dti_features])
dti_covars = dti.drop(columns=dti_features)
print(f"Array of {dti_array.shape} and covars of {dti_covars.shape}")

Array of (152, 1) and covars of (152, 6)


In [201]:
anat_model, anat_adj = harmonizationLearn(anat_array, anat_covars)
dti_model, dti_adj = harmonizationLearn(dti_array, dti_covars)


[neuroHarmonize]: Bypassing empirical Bayes step because only one variable to harmonize.

[neuroHarmonize]: Bypassing empirical Bayes step because only one variable to harmonize.


In [209]:
dti_adj

array([[nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
       [nan],
      

In [210]:
harm.loc[harm['PVSVF'].notna(), 'PVSVF-HARM'] = anat_adj
harm.loc[harm['ALPS'].notna(), 'ALPS-HARM'] = dti_adj
harm.to_csv(f"{root}/harm.csv")
harm

,SITE,SD,AGE,SEX,KSS,CHRONO,PVSVF,ALPS,PVSVF-HARM,ALPS-HARM
SUB,,,,,,,,,,
sub-JU105006,Juelich,0,26.063014,0,3.0,3.0,0.000291,NaN,NaN,NaN
sub-JU105006,Juelich,1,26.063014,0,8.0,3.0,0.000283,NaN,NaN,NaN
sub-JU105009,Juelich,0,22.808219,1,3.0,3.0,0.000206,NaN,NaN,NaN
sub-JU105009,Juelich,1,22.808219,1,5.0,3.0,0.000205,NaN,NaN,NaN
sub-JU105014,Juelich,0,26.265753,0,4.0,3.0,0.000346,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
sub-ME322,Melbourne,1,23.000000,0,4.0,1.0,0.001028,NaN,NaN,NaN
sub-ME323,Melbourne,0,21.000000,0,2.0,3.0,0.000752,NaN,NaN,NaN
sub-ME323,Melbourne,1,21.000000,0,5.0,3.0,0.000774,NaN,NaN,NaN
